In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv


# Netflix ETL Data Pipeline
**Author:** Yash Pareek | 2022BTECH115 | JK Lakshmipat University  
**Tools:** Python, Pandas, SQLite, Plotly  

### Pipeline Flow
Extract (CSV) → Transform (Clean + Features) → Load (SQLite + Charts)

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

warnings.filterwarnings('ignore')
print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
# EXTRACT
df = pd.read_csv('/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv')
print(f"Records loaded: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"\nNull values:\n{df.isnull().sum()}")
df.head()

Records loaded: 8807
Columns: 12

Null values:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [5]:
# TRANSFORM
df_clean = df.copy()

# 1. Remove duplicates
df_clean.drop_duplicates(subset='show_id', inplace=True)

# 2. Handle nulls
df_clean['director'].fillna('Unknown', inplace=True)
df_clean['cast'].fillna('Unknown', inplace=True)
df_clean['country'].fillna('Unknown', inplace=True)
df_clean['rating'].fillna('Not Rated', inplace=True)
df_clean.dropna(subset=['date_added', 'duration'], inplace=True)

# 3. Fix data types
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'].str.strip())
df_clean['release_year'] = df_clean['release_year'].astype(int)

# 4. New features
df_clean['year_added'] = df_clean['date_added'].dt.year
df_clean['month_added'] = df_clean['date_added'].dt.month
df_clean['duration_minutes'] = df_clean.apply(
    lambda x: int(x['duration'].split(' ')[0]) if 'min' in str(x['duration']) else None, axis=1
)
df_clean['season_count'] = df_clean.apply(
    lambda x: int(x['duration'].split(' ')[0]) if 'Season' in str(x['duration']) else None, axis=1
)
df_clean['primary_country'] = df_clean['country'].apply(
    lambda x: x.split(',')[0].strip()
)
df_clean['primary_genre'] = df_clean['listed_in'].apply(
    lambda x: x.split(',')[0].strip()
)

print(f"Records after cleaning: {df_clean.shape[0]}")
print(f"Total features: {df_clean.shape[1]}")
df_clean.head()

Records after cleaning: 8794
Total features: 18


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_minutes,season_count,primary_country,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,9,90.0,NaN,United States,Documentaries
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,9,NaN,2.0,South Africa,International TV Shows
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,9,NaN,1.0,Unknown,Crime TV Shows
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,9,NaN,1.0,Unknown,Docuseries
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,9,NaN,2.0,India,International TV Shows


In [6]:
# LOAD - Store in SQLite Database
conn = sqlite3.connect('/kaggle/working/netflix_etl.db')

df_clean.to_sql('netflix_transformed', conn, if_exists='replace', index=False)

# Verify
query = pd.read_sql_query("SELECT type, COUNT(*) as total FROM netflix_transformed GROUP BY type", conn)
print("=== DATA LOADED TO SQLite ===")
print(f"Database: netflix_etl.db")
print(f"Table: netflix_transformed")
print(f"\nContent breakdown:")
print(query)
conn.close()


=== DATA LOADED TO SQLite ===
Database: netflix_etl.db
Table: netflix_transformed

Content breakdown:
      type  total
0    Movie   6128
1  TV Show   2666


In [7]:
# ANALYZE - Charts
# Chart 1: Movies vs TV Shows
type_counts = df_clean['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

fig = go.Figure(go.Pie(
    labels=type_counts['type'],
    values=type_counts['count'],
    hole=0.4,
    marker=dict(colors=['#E50914', '#221f1f'])
))
fig.update_layout(title='Content Distribution: Movies vs TV Shows', height=400)
fig.show()

# Chart 2: Content added per year
yearly = df_clean.groupby(['year_added', 'type']).size().reset_index(name='count')
yearly = yearly[yearly['year_added'] >= 2013]

fig2 = px.bar(
    yearly, x='year_added', y='count', color='type',
    color_discrete_map={'Movie': '#E50914', 'TV Show': '#564d4d'},
    title='Content Added to Netflix Per Year',
    barmode='group'
)
fig2.show()

# Chart 3: Top 10 countries
top_countries = df_clean[df_clean['primary_country'] != 'Unknown']['primary_country'] \
    .value_counts().head(10).reset_index()
top_countries.columns = ['country', 'count']

fig3 = px.bar(
    top_countries, x='count', y='country', orientation='h',
    title='Top 10 Countries by Netflix Content',
    color='count', color_continuous_scale=['#564d4d', '#E50914']
)
fig3.update_layout(yaxis={'categoryorder': 'total ascending'})
fig3.show()

# Chart 4: Top Genres
all_genres = ', '.join(df_clean['listed_in'].dropna()).split(', ')
genre_counts = Counter(all_genres).most_common(10)
genre_df = pd.DataFrame(genre_counts, columns=['genre', 'count'])

fig4 = px.bar(
    genre_df, x='count', y='genre', orientation='h',
    title='Top 10 Genres on Netflix',
    color='count', color_continuous_scale=['#564d4d', '#E50914']
)
fig4.update_layout(yaxis={'categoryorder': 'total ascending'})
fig4.show()

In [8]:
# PIPELINE SUMMARY
print("==========================================")
print("   NETFLIX ETL PIPELINE - FINAL SUMMARY   ")
print("==========================================")
print(f"\nEXTRACT")
print(f"  Source          : netflix_titles.csv")
print(f"  Records loaded  : {df.shape[0]}")
print(f"  Columns         : {df.shape[1]}")
print(f"\nTRANSFORM")
print(f"  Duplicates removed : {df.shape[0] - df_clean.shape[0]}")
print(f"  Nulls handled      : Yes")
print(f"  New features added : 6")
print(f"  Final records      : {df_clean.shape[0]}")
print(f"  Final columns      : {df_clean.shape[1]}")
print(f"\nLOAD")
print(f"  Database        : netflix_etl.db (SQLite)")
print(f"  Table           : netflix_transformed")
print(f"  Movies loaded   : 6128")
print(f"  TV Shows loaded : 2666")
print(f"\nVISUALIZATIONS")
print(f"  Charts generated : 4")
print(f"\nPipeline completed successfully!")

   NETFLIX ETL PIPELINE - FINAL SUMMARY   

EXTRACT
  Source          : netflix_titles.csv
  Records loaded  : 8807
  Columns         : 12

TRANSFORM
  Duplicates removed : 13
  Nulls handled      : Yes
  New features added : 6
  Final records      : 8794
  Final columns      : 18

LOAD
  Database        : netflix_etl.db (SQLite)
  Table           : netflix_transformed
  Movies loaded   : 6128
  TV Shows loaded : 2666

VISUALIZATIONS
  Charts generated : 4

Pipeline completed successfully!
